# Codeberg Scraping – Blinde Signaturen


In [1]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv
from tqdm import tqdm
from urllib.parse import quote

load_dotenv()

CODEBERG_TOKEN = os.getenv("CODEBERG_TOKEN")  # optional

HEADERS = {}
if CODEBERG_TOKEN:
    HEADERS["Authorization"] = f"token {CODEBERG_TOKEN}"

API_BASE = "https://codeberg.org/api/v1"

# Codeberg-Suche durchsucht name, description und path (NICHT das README).
SEARCH_TERMS = [
    '"blind signatures"',
    '"anonymous credentials"',
    '"blind"',
    '"blind RSA"',
    '"blind token"',

    '"partially blind signature"',
    '"blind schnorr"',
    '"blind BLS"',                      #Boneh-Lynn-Shacham
    '"VOPRF"',
    '"OPRF" "token"',
    '"RSA blind signatures"',

    '"anonymous tokens"',
    '"private access token"',
    '"trust token"',
    '"private state token"',
    '"unlinkable token"',
    '"challenge bypass"',

    '"blinded message"',
    '"blind issuance"',
]

# Dieselben Pflichtbegriffe wie beim GitHub-Scraping
REQUIRED_TERMS = [
    "blind signature",
    "blind signatures",
    "blind signing",
    "blind sign",
    "blind rsa",
    "blind schnorr",
    "blinded message",
    "blinding factor",
    "blind issuance",
    "unblind",
    "partially blind",
    "voprf",
    "anonymous token",
    "private access token",
    "chaumian",
]

GITHUB_FILE  = "blind_signature_repos_v2_final.xlsx"   # bestehende GitHub-Ergebnisse
OUTPUT_FILE  = "codeberg_only_repos_v2.xlsx"
MIN_STARS    = 0   # Codeberg-Projekte haben generell weniger Stars als GitHub


In [2]:
def search_projects(query, limit=50, max_pages=10):
    all_items = []

    for page in range(1, max_pages + 1):
        params = {
            "q": query.replace('"', ''),
            "page": page,
            "limit": limit
        }

        response = requests.get(
            f"{API_BASE}/repos/search",
            headers=HEADERS,
            params=params
        )

        if response.status_code != 200:
            print("Search Error:", response.status_code, response.text[:200])
            break

        data = response.json()
        items = data.get("data", [])

        if not items:
            break

        all_items.extend(items)

        if len(items) < limit:
            break

        time.sleep(0.5)

    return all_items


def get_readme(owner, repo):
    candidates = ["README.md", "README.rst", "README.txt", "README", "readme.md"]

    branches = ["main", "master"]

    for branch in branches:
        for filename in candidates:
            url = f"{API_BASE}/repos/{owner}/{repo}/raw/branch/{branch}/{filename}"

            response = requests.get(url, headers=HEADERS)

            if response.status_code == 200:
                return response.text

    return ""


def is_relevant(readme_text):
    """True wenn mindestens ein Pflichtbegriff im README vorkommt."""
    t = readme_text.lower()
    return any(term in t for term in REQUIRED_TERMS)


def classify_readme(text):
    t = text.lower()
    found = [term for term in REQUIRED_TERMS if term in t]
    return {
        "matched_terms":          ", ".join(found),
        "mentions_anti_bot":      any(k in t for k in ["captcha", "bot detection", "challenge bypass", "trust token", "private state token"]),
        "mentions_fraud_prevent": any(k in t for k in ["fraud prevention", "fraud detection", "private reporting", "touchpoint"]),
        "mentions_credentials":   any(k in t for k in ["anonymous credential", "anoncred", "de-identified", "selective disclosure"]),
        "mentions_advertising":   any(k in t for k in ["adveil", "ad measurement", "privacy-preserving ad", "anonymous ad"]),
        "mentions_privacy":       "privacy" in t,
        "mentions_authentication": "authentication" in t,
    }


In [3]:
# ── Codeberg durchsuchen ───────────────────────────────────────────────────────

seen_ids = set()
all_rows = []
skipped = 0

for term in SEARCH_TERMS:
    projects = search_projects(term)

    for proj in tqdm(projects, desc=term):
        pid = proj["id"]

        if pid in seen_ids:
            continue
        seen_ids.add(pid)

        if proj.get("star_count", 0) < MIN_STARS:
            continue

        readme = get_readme(pid, proj.get("default_branch"))

        if not is_relevant(readme):
            skipped += 1
            time.sleep(0.5)
            continue

        analysis = classify_readme(readme)

        row = {
            "search_term": term,
            "repo_name":   proj["path"],
            "owner":       proj["namespace"]["path"] if proj.get("namespace") else "",
            "stars":       proj.get("star_count", 0),
            "forks":       proj.get("forks_count", 0),
            "created_at":  proj.get("created_at", ""),
            "updated_at":  proj.get("last_activity_at", ""),
            "description": proj.get("description", ""),
            "topics":      ", ".join(proj.get("topics", [])),
            "repo_url":    proj["web_url"],
            "platform":    "codeberg",
            "readme":      readme[:5000],
        }

        row.update(analysis)
        all_rows.append(row)
        time.sleep(0.5)

df_codeberg = pd.DataFrame(all_rows)
print(f"\nCodeberg: {len(df_codeberg)} relevante Repos gefunden, {skipped} gefiltert")

"blind signatures": 0it [00:00, ?it/s]
"anonymous credentials": 0it [00:00, ?it/s]
"blind": 100%|██████████| 57/57 [01:32<00:00,  1.63s/it]
"blind RSA": 0it [00:00, ?it/s]
"blind token": 0it [00:00, ?it/s]
"partially blind signature": 0it [00:00, ?it/s]
"blind schnorr": 0it [00:00, ?it/s]
"blind BLS": 0it [00:00, ?it/s]
"VOPRF": 0it [00:00, ?it/s]
"OPRF" "token": 0it [00:00, ?it/s]
"RSA blind signatures": 0it [00:00, ?it/s]
"anonymous tokens": 0it [00:00, ?it/s]
"private access token": 0it [00:00, ?it/s]
"trust token": 0it [00:00, ?it/s]
"private state token": 0it [00:00, ?it/s]
"unlinkable token": 0it [00:00, ?it/s]
"challenge bypass": 0it [00:00, ?it/s]
"blinded message": 0it [00:00, ?it/s]
"blind issuance": 0it [00:00, ?it/s]


Codeberg: 0 relevante Repos gefunden, 57 gefiltert


In [ ]:
# ── Mit GitHub-Ergebnissen vergleichen ───────────────────────────────────────

if df_codeberg.empty:
    print("Keine relevanten Repos gefunden – Vergleich mit GitHub wird übersprungen.")
    df_codeberg = pd.DataFrame(columns=[
        "search_term", "repo_name", "owner", "stars", "forks", "language",
        "created_at", "updated_at", "description", "topics", "repo_url",
        "platform", "readme", "matched_terms", "mentions_anti_bot",
        "mentions_fraud_prevent", "mentions_credentials",
        "mentions_advertising", "mentions_privacy", "mentions_authentication",
        "overlap"
    ])
else:
    df_github = pd.read_excel(GITHUB_FILE)

    # Vergleichsschlüssel: normalisierter Repo-Name + normalisierter Owner.
    # Da viele Projekte auf GitHub UND Codeberg gespiegelt werden (gleicher Name,
    # teils gleicher Owner), prüfen wir zwei Stufen:
    #   1. Exakter Match: owner/repo_name identisch
    #   2. Name-Match: nur repo_name identisch (mögliche Mirrors mit anderem Owner)

    def norm(s):
        return str(s).lower().strip().replace("-", "").replace("_", "")

    github_full  = set(norm(o) + "/" + norm(r) for o, r in zip(df_github["owner"], df_github["repo_name"]))
    github_names = set(norm(r) for r in df_github["repo_name"])

    def overlap_status(row):
        full = norm(row["owner"]) + "/" + norm(row["repo_name"])
        if full in github_full:
            return "exact"          # identisches Projekt (Mirror)
        if norm(row["repo_name"]) in github_names:
            return "name_match"     # gleicher Name, anderer Owner → manuell prüfen
        return "unique"

    df_codeberg["overlap"] = df_codeberg.apply(overlap_status, axis=1)

    n_exact  = (df_codeberg["overlap"] == "exact").sum()
    n_name   = (df_codeberg["overlap"] == "name_match").sum()
    n_unique = (df_codeberg["overlap"] == "unique").sum()

    print(f"Exakte Überschneidungen (Mirrors):  {n_exact}")
    print(f"Namens-Übereinstimmungen (prüfen): {n_name}")
    print(f"Nur auf Codeberg:                   {n_unique}")

Keine relevanten Repos gefunden – Vergleich mit GitHub wird übersprungen.


In [5]:
# ── Nur nicht-überschneidende Repos exportieren ──────────────────────────────

# 'unique' sicher nicht auf GitHub; 'name_match' zur manuellen Prüfung mit ausgeben
df_out = df_codeberg[df_codeberg["overlap"].isin(["unique", "name_match"])].copy()
df_out = df_out.sort_values(["overlap", "stars"], ascending=[False, False]).reset_index(drop=True)

df_out.to_excel(OUTPUT_FILE, index=False)

print(f"Gespeichert: {OUTPUT_FILE} ({len(df_out)} Repos)")

df_out[["overlap", "owner", "repo_name", "stars", "description"]].head(20)

Gespeichert: codeberg_only_repos_v2.xlsx (0 Repos)


,overlap,owner,repo_name,stars,description
